In [ ]:
# Cell 1: Mount Google Drive so the notebook can read the dataset
# and save the trained model weights permanently

from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive

Mounted at /content/drive
 best.pt	     'Colab Notebooks'		        NVD
 car_detection_snow   FireExtinguisher.v1i.yolov8.zip   nvd_dataset


In [ ]:
# Cell 2: Extract the Roboflow YOLOv8 dataset (FireExtinguisher.v1i.yolov8.zip)
# into the local Colab disk. Expected result: train/, valid/ and data.yaml.
# The -o flag overwrites existing files without asking.

!mkdir -p /content/dataset
!unzip -q "/content/drive/MyDrive/FireExtinguisher.v1i.yolov8.zip" -d /content/dataset
!ls /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
data.yaml  README.roboflow.txt	train  valid


In [5]:
# Cell 3: Install Ultralytics (YOLOv8 framework) and write the dataset config.
# The config tells YOLO where the training and validation images are
# and defines a single class: extinguisher.

!pip install -q ultralytics

yaml_text = """
path: /content/dataset
train: train/images
val: valid/images
names:
  0: extinguisher
"""
open('/content/dataset/my_data.yaml', 'w').write(yaml_text)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/my_data.yaml'

In [ ]:
# Cell 4: Fine-tune a COCO-pretrained YOLOv8s model on the fire extinguisher
# dataset (transfer learning). Source of pretrained weights: Ultralytics.
from ultralytics import YOLO

# Load pretrained YOLOv8 small: a good trade-off between speed and accuracy
model = YOLO("yolov8s.pt")

model.train(
    data="/content/dataset/my_data.yaml",  # dataset config from Cell 3
    epochs=50,        # maximum number of training passes over the data
    imgsz=640,        # input image size (pixels)
    batch=16,         # images processed per step
    device=0,         # use the first GPU (T4)
    patience=15,      # early stopping: stop if no improvement for 15 epochs
    project="/content/drive/MyDrive/r7020e_runs",  # save results to Drive
    name="fire_ext_v8s",  # run folder name
    plots=True,       # save training curves and confusion matrix
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/my_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d81697bb000>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Cell 5: Second training run with YOLO26s for comparison with YOLOv8s.
# Same dataset, same hyperparameters, so the only difference is the model.
# Pretrained weights source: Ultralytics (COCO).
from ultralytics import YOLO

# Load pretrained YOLO26 small
model = YOLO("yolo26s.pt")

model.train(
    data="/content/dataset/my_data.yaml",  # same dataset config as before
    epochs=50,        # same as the YOLOv8s run, for a fair comparison
    imgsz=640,        # input image size (pixels)
    batch=16,         # images processed per step
    device=0,         # use the GPU (T4)
    patience=15,      # early stopping
    project="/content/drive/MyDrive/r7020e_runs",  # save results to Drive
    name="fire_ext_26s",  # separate run folder, does not overwrite the YOLOv8s run
    plots=True,       # training curves and confusion matrix
)

Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/my_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fire_ext_26s, nbs=64, nms=None, opset=None, optimize=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d816084de80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [6]:
# Run the trained detector on the colour frames and save boxes + JSON
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics==8.4.182

import json
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = "/content/drive/MyDrive/r7020e_runs/fire_ext_26s/weights/best.pt"  # main model
DATA = "/content/drive/MyDrive/r7020e_data"   # folder that contains eval/ and test/ with the colour images
CONF = 0.25
IOU = 0.45   # NMS threshold: removes duplicate overlapping boxes

model = YOLO(WEIGHTS)

for split in ["eval", "test"]:
    imgs = sorted(Path(f"{DATA}/{split}").rglob("*.png"))
    print(split, len(imgs), "images")
    results = model.predict([str(p) for p in imgs], conf=CONF, iou=IOU, imgsz=640, verbose=False,
                            save=True, project="/content/drive/MyDrive/r7020e_runs/infer", name=split, exist_ok=True)
    out = []
    for p, r in zip(imgs, results):
        dets = [{"det_id": f"{p.stem}_{i}",
                 "bbox_xyxy": [round(v, 1) for v in r.boxes.xyxy[i].tolist()],
                 "confidence": round(float(r.boxes.conf[i]), 4),
                 "class": "extinguisher"} for i in range(len(r.boxes))]
        out.append({"image_id": p.stem, "image_path": str(p), "detections": dets})
    json.dump(out, open(f"/content/drive/MyDrive/r7020e_runs/infer/detections_{split}.json", "w"), indent=1)
    print(split, "detections:", sum(len(o["detections"]) for o in out))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
eval 127 images
Results saved to /content/drive/MyDrive/r7020e_runs/infer/eval
eval detections: 301
test 88 images
Results saved to /content/drive/MyDrive/r7020e_runs/infer/test
test detections: 150
